# ResStock models: select, download, simulate, build load tables

**Purpose:** pick ResStock buildings matched to AMI meters, download and run their OpenStudio models, then build 15-min / hourly / annual load tables for the optimization (alongside the AMI data). Works for the baseline (`UPGRADE = 0`) or any other ResStock upgrade (e.g. `UPGRADE = 4`): set `UPGRADE` in the "Upgrade selection" cell and all inputs, S3 paths and outputs follow.

## Workflow overview
Outputs shown for the baseline; for upgrade `N` every name gets an `_upNN` tag (e.g. `resstock_models_up04/`, `building_ids_metadata_2024_up04.csv`, `resstock_timeseries_all_15_up04.csv`).

| # | Section | Key output |
|---|---------|-----------|
| 1 | Select building IDs (NY metadata filtered to Tompkins and neighboring counties; for upgrades, only applicable buildings) | `data/base_files/building_ids_metadata_2024.csv` |
| 2 | Download ResStock models from S3 (parallel) and prep OSW, measures and weather | `data/resstock_models/bldgXXXXXXX/` |
| 3 | Run OpenStudio simulations (parallel) | `eplusout.csv` per building |
| 4 | Convert simulation output to 15-min / hourly / annual tables (parallel) | `data/parquet/resstock_timeseries_*` |
| 5 | Caveats and reference | - |

## Related notebooks
- `resstock_timeseries_download.ipynb`: shortcut that downloads the released ResStock timeseries directly (skip sections 2-4 of this notebook if you only need the load timeseries). Baseline only; it does not read `UPGRADE`.
- `ami_data_prep.ipynb`: prepares the AMI data (2024, duplicates averaged, Feb 29 removed).
- `resstock_metadata_plots.ipynb`: metadata distribution plots for the matched ResStock buildings.

## Notes
- Run top to bottom.
- Windows only: edit `openstudio_path` in section 3 if OpenStudio 3.8.0 is installed elsewhere.
- Sections 2 and 3 use `building_ids` from cell 2a; sections 2-4 use `extract_dir` and the output paths from the "Upgrade selection" cell.


## Setup
Assumes the project environment is already created and activated (`python -m pip install -r requirements.txt`). The two setup cells below are the same in every notebook of this project; run them first. Run notebooks from `grid_edge_fy25/notebooks/` (all paths are relative to it).

In [1]:
# Preflight check: assumes environment already created from requirements.txt
import importlib.util
import shutil
import subprocess
import sys

required_modules = [
    "pandas",
    "numpy",
    "matplotlib",
    "pyarrow",
    "fastparquet",
    "openpyxl",
    "boto3",
]

missing = [m for m in required_modules if importlib.util.find_spec(m) is None]

if missing:
    raise RuntimeError(
        "Missing Python packages: " + ", ".join(missing) +
        "\nCreate/activate the project environment and run: python -m pip install -r requirements.txt"
    )

# Check awscli availability (prefer module invocation for consistency)
try:
    result = subprocess.run(
        [sys.executable, "-m", "awscli", "--version"],
        check=True,
        capture_output=True,
        text=True,
    )
    print(result.stdout.strip() or result.stderr.strip())
except Exception as exc:
    aws_on_path = shutil.which("aws")
    if aws_on_path:
        print("aws executable found on PATH:", aws_on_path)
    else:
        raise RuntimeError(
            "awscli is not available in this environment. Install from requirements.txt."
        ) from exc

print("Preflight OK: environment is ready.")

aws-cli/1.42.0 Python/3.13.15 Windows/11 botocore/1.40.0
Preflight OK: environment is ready.


In [2]:
# Common setup (identical in all notebooks of this project)
import ast
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# Paths (relative to grid_edge_fy25/notebooks/)
current_path = Path(os.getcwd())
data_path = current_path / ".." / "data"
base_files = data_path / "base_files"        # building id lists, measures, weather, workflow
input_files = data_path / "input_files"      # NY metadata, AMI consumption
parquet_folder = data_path / "parquet"       # load tables written for the optimization
results_path = current_path / ".." / "results"
extract_dir = data_path / "resstock_models"  # bldgXXXXXXX folders (models + simulation output)

parquet_folder.mkdir(parents=True, exist_ok=True)


---
## Upgrade selection
Set `UPGRADE` once; every path below is derived from it. `0` is the baseline and keeps the original file and folder names. Any other id adds an `_upNN` tag to every output (model folder, building-ID list, load tables), so runs for different upgrades never overwrite each other. The upgrade metadata file must be in `data/input_files/` as `NY_upgradeNN_metadata_and_annual_results_2024_amy.csv` (baseline: `NY_baseline_...`), and only buildings with `applicability == True` in it are selected (upgrade zips exist only for those).

Re-run this cell and everything after it after changing `UPGRADE`. Upgrade 4 = "ENERGY STAR heat pump with existing system as backup".

In [3]:
# ========================
# UPGRADE SELECTION: the only value to change to run another ResStock upgrade
# ========================
UPGRADE = 4          # 0 = baseline, 4 = ENERGY STAR heat pump with existing system as backup, ...

assert isinstance(UPGRADE, int) and UPGRADE >= 0, "UPGRADE must be a non-negative integer"
UPGRADE_TAG = "" if UPGRADE == 0 else f"_up{UPGRADE:02d}"   # baseline keeps the original names

# --- Inputs ---
METADATA_PATH = input_files / (
    "NY_baseline_metadata_and_annual_results_2024_amy.csv" if UPGRADE == 0
    else f"NY_upgrade{UPGRADE:02d}_metadata_and_annual_results_2024_amy.csv"
)
# --- S3 source (AMY release 2): upgrade=N/ folder, bldgXXXXXXX-upNN.zip files ---
S3_BASE = ("s3://oedi-data-lake/nrel-pds-building-stock/"
           "end-use-load-profiles-for-us-building-stock/2024/"
           "resstock_amy2018_release_2/model_and_schedule_files/"
           f"building_energy_models/upgrade={UPGRADE}/")
MODEL_ZIP_SUFFIX = f"-up{UPGRADE:02d}.zip"

# --- Outputs (all carry UPGRADE_TAG) ---
BUILDING_IDS_CSV = base_files / f"building_ids_metadata_2024{UPGRADE_TAG}.csv"
extract_dir = data_path / f"resstock_models{UPGRADE_TAG}"    # bldgXXXXXXX folders (models + simulation output)
OUT_15_CSV  = parquet_folder / f"resstock_timeseries_all_15{UPGRADE_TAG}.csv"
OUT_15_PQ   = parquet_folder / f"resstock_timeseries_all_15{UPGRADE_TAG}.parquet"
OUT_HR_CSV  = parquet_folder / f"resstock_timeseries_all_hour{UPGRADE_TAG}.csv"
OUT_HR_PQ   = parquet_folder / f"resstock_timeseries_all_hour{UPGRADE_TAG}.parquet"
OUT_ANL_CSV = parquet_folder / f"resstock_timeseries_annual{UPGRADE_TAG}.csv"
OUT_ANL_PQ  = parquet_folder / f"resstock_timeseries_annual{UPGRADE_TAG}.parquet"

if not METADATA_PATH.exists():
    raise FileNotFoundError(f"Metadata for upgrade {UPGRADE} not found: {METADATA_PATH}")

print(f"UPGRADE = {UPGRADE} (tag '{UPGRADE_TAG}')")
print("Metadata   :", METADATA_PATH.name)
print("S3 source  :", S3_BASE, f"bldgXXXXXXX{MODEL_ZIP_SUFFIX}")
print("Building ID:", BUILDING_IDS_CSV.name)
print("Models dir :", extract_dir)
print("Load tables:", OUT_15_CSV.name, "/", OUT_HR_CSV.name, "/", OUT_ANL_CSV.name)


UPGRADE = 4 (tag '_up04')
Metadata   : NY_upgrade04_metadata_and_annual_results_2024_amy.csv
S3 source  : s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_amy2018_release_2/model_and_schedule_files/building_energy_models/upgrade=4/ bldgXXXXXXX-up04.zip
Building ID: building_ids_metadata_2024_up04.csv
Models dir : c:\Users\jxiong\Documents\GitHub\grid_edge_ithaca\grid_edge_fy25\notebooks\..\data\resstock_models_up04
Load tables: resstock_timeseries_all_15_up04.csv / resstock_timeseries_all_hour_up04.csv / resstock_timeseries_annual_up04.csv


---
## 1. Select building IDs
Reads the NY ResStock metadata for the selected `UPGRADE` (`METADATA_PATH`), keeps only Cayuga, Chemung, Cortland, Schuyler, Seneca, Tioga and Tompkins counties (and, for upgrades, only buildings where `applicability` is True), and saves the `bldg_id` list to `BUILDING_IDS_CSV`. If that file already exists it is reused; delete it to regenerate.


In [4]:
# SCRIPT TO READ AND SAVE ALL BUILDING IDS FROM THE NY METADATA FILE (FILTERED FOR DRYDEN RELEVANT COUNTIES)
# Paths come from the "Upgrade selection" cell: METADATA_PATH (input), BUILDING_IDS_CSV (output)

output_path = BUILDING_IDS_CSV
buildstock_metadata_path = METADATA_PATH

df = pd.read_csv(buildstock_metadata_path)

selected_counties = [
    "Cayuga County",
    "Chemung County",
    "Cortland County",
    "Schuyler County",
    "Seneca County",
    "Tioga County",
    "Tompkins County",
]

df = df[df["in.county_name"].isin(selected_counties)]

# Upgrade models exist only for buildings the upgrade applies to (all True for the baseline)
if "applicability" in df.columns:
    n_county = len(df)
    df = df[df["applicability"].astype(bool)]
    print(f"Upgrade {UPGRADE}: {len(df)} of {n_county} county buildings are applicable")

if not output_path.exists():

    # Number of rows in the DataFrame
    num_rows = len(df)
    print("Number of rows:", num_rows)

    building_ids_metadata = df["bldg_id"]

    # Save building IDs to CSV

    building_ids_metadata.to_csv(output_path, index=False)

else:
    
    building_ids_metadata = pd.read_csv(output_path)
    print(f"Using existing {output_path.name} (delete it to regenerate)")

print("bldg_id count:", len(building_ids_metadata))
print(building_ids_metadata)
print('done')

df


C:\Users\jxiong\AppData\Local\Temp\1\ipykernel_3572\1555914867.py:7: DtypeWarning: Columns (133) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(buildstock_metadata_path)


Upgrade 4: 483 of 766 county buildings are applicable
Using existing building_ids_metadata_2024_up04.csv (delete it to regenerate)
bldg_id count: 483
     bldg_id
0        510
1       1700
2       3424
3       6034
4       6232
..       ...
478   543959
479   547188
480   547386
481   547426
482   548817

[483 rows x 1 columns]
done


,bldg_id,upgrade,weight,upgrade_name,applicability,in.sqft,in.representative_income,in.ahs_region,in.aiannh_area,in.area_median_income,...,upgrade.cooling_setpoint_offset_magnitude,upgrade.cooling_setpoint_offset_period,upgrade.heating_setpoint_has_offset,upgrade.heating_setpoint_offset_magnitude,upgrade.heating_setpoint_offset_period,upgrade.hvac_cooling_efficiency,upgrade.hvac_cooling_partial_space_conditioning,upgrade.hvac_heating_efficiency,upgrade.hvac_secondary_heating_efficiency,upgrade.hvac_secondary_heating_fuel
33,510,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,2179,47457.0,Non-CBSA Middle Atlantic,No,60-80%,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
110,1700,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,3310,109096.0,Non-CBSA Middle Atlantic,No,150%+,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
210,3424,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,3310,280058.0,Non-CBSA Middle Atlantic,No,150%+,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
382,6034,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,1698,0.0,Non-CBSA Middle Atlantic,No,Not Available,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
399,6232,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,2179,270137.0,Non-CBSA Middle Atlantic,No,150%+,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33426,543959,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,2678,273335.0,Non-CBSA Middle Atlantic,No,150%+,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
33605,547188,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,1698,42401.0,Non-CBSA Middle Atlantic,No,30-60%,...,0F,NaN,No,0F,NaN,Non-Ducted Heat Pump,100% Conditioned,"MSHP, SEER 16, 9.2 HSPF, Separate Backup, Max ...","Fuel Wall/Floor Furnace, 68% AFUE",Natural Gas
33620,547386,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,1698,89096.0,Non-CBSA Middle Atlantic,No,120-150%,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN
33621,547426,4,252.301639,ENERGY STAR heat pump with existing system as ...,True,1228,22473.0,Non-CBSA Middle Atlantic,No,30-60%,...,NaN,NaN,NaN,NaN,NaN,Ducted Heat Pump,100% Conditioned,"Dual-Fuel ASHP, SEER 16, 9.2 HSPF, Integrated ...",NaN,NaN


---
## 2. Download ResStock models
First load `building_ids` (2a), then run the parallel downloader (2b). It downloads `bldgXXXXXXX-upNN.zip` from the `upgrade=N/` S3 folder (`NN` = `UPGRADE`, two digits), unzips it to `data/resstock_models[_upNN]/bldgXXXXXXX/models`, and copies in weather, measures and a patched `workflow_resstock.osw`.

**2a.** Load `building_ids` from the saved building-ID CSV for the selected upgrade (county-filtered, applicable buildings only).

**Weather basis: AMY throughout.** The metadata (`NY_..._2024_amy.csv`) and the models (`resstock_amy2018_release_2`) must come from the same release, so `S3_BASE` (set in the "Upgrade selection" cell) points at the AMY models. The simulations still run with `Ithaca_2024.epw`. Each upgrade has its own model folder, so baseline and upgrade models never mix. If `data/resstock_models/` already contains models from the old `resstock_tmy3_release_2` release, delete those `bldgXXXXXXX` folders first: the downloader skips folders that already have a `models` directory.


In [5]:
# SCRIPT TO READ AND STORE ALL BUILDING IDS FOR THE SELECTED UPGRADE (FILTERED FOR DRYDEN RELEVANT COUNTIES)
# SCRIPT STORES LIST OF BUILDING IDS
df_ids = pd.read_csv(BUILDING_IDS_CSV)
building_ids = df_ids["bldg_id"]
print(f"Upgrade {UPGRADE}: bldg_id count:", len(building_ids), f"({BUILDING_IDS_CSV.name})")


Upgrade 4: bldg_id count: 483 (building_ids_metadata_2024_up04.csv)


**2b. Parallel downloader (boto3 multipart, falls back to AWS CLI).** Downloads every building in `BUILDING_IDS`, which is set from `building_ids` (2a). To run a subset, overwrite `BUILDING_IDS` in the CONFIG block; tune `MAX_WORKERS` for your bandwidth/CPU. Skips buildings already downloaded.

In [6]:
# --- Faster ResStock downloader: parallel + multipart S3 ---
import os, json, zipfile, sys, subprocess, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# ========================
# CONFIG
# ========================
BUILDING_IDS = [int(b) for b in building_ids]   # from cell 2a; overwrite with a list to run a subset
# S3_BASE (upgrade=N/ folder), MODEL_ZIP_SUFFIX (-upNN.zip) and extract_dir (resstock_models[_upNN])
# all come from the "Upgrade selection" cell.

BASE_FILES = Path("../data/base_files")
MEASURES_OG = BASE_FILES / "measures"
WORKFLOW_OG = BASE_FILES / "workflow_resstock.osw"
WEATHER_FILE = "Ithaca_2024.epw"
WEATHER_OG = BASE_FILES / "weather" / WEATHER_FILE

EXTRACT_DIR = Path(extract_dir)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 6          # try 4–8 depending on your bandwidth/CPU
CHUNK_MB = 8             # multipart chunk size for S3 (MB)

# ========================
# S3 DOWNLOAD IMPL (boto3 if available -> fast; else awscli module)
# ========================
_use_boto3 = False
try:
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config
    from boto3.s3.transfer import TransferConfig, S3Transfer
    _use_boto3 = True
    _s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
    _transfer = S3Transfer(
        _s3,
        TransferConfig(
            multipart_threshold=CHUNK_MB*1024*1024,
            multipart_chunksize=CHUNK_MB*1024*1024,
            max_concurrency=10,
            use_threads=True
        )
    )
except Exception:
    pass

AWS_PREFIX = [sys.executable, "-m", "awscli"]  # fallback

def s3_url_parts(s3_url: str):
    # "s3://bucket/key..." -> ("bucket","key")
    assert s3_url.startswith("s3://")
    rest = s3_url[5:]
    bucket, key = rest.split("/", 1)
    return bucket, key

def download_s3(s3_url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if _use_boto3:
        bucket, key = s3_url_parts(s3_url)
        _transfer.download_file(bucket, key, str(dest))
    else:
        cmd = AWS_PREFIX + ["s3", "cp", s3_url, str(dest), "--no-sign-request", "--only-show-errors"]
        subprocess.run(cmd, check=True)

# ========================
# UTILITIES
# ========================
def ensure_base_files():
    for p in [MEASURES_OG, WORKFLOW_OG, WEATHER_OG]:
        if not p.exists():
            raise FileNotFoundError(f"Missing base file/dir: {p}")

def unzip_to_dir(zip_path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)

def copy_weather_and_measures(local_extract_path: Path):
    (local_extract_path / "weather").mkdir(exist_ok=True)
    shutil.copy(WEATHER_OG, local_extract_path / "weather")

    measures_dst = local_extract_path / "measures"
    if measures_dst.exists():
        shutil.rmtree(measures_dst)
    shutil.copytree(MEASURES_OG, measures_dst)

def patch_osw(local_extract_path: Path):
    osw_dst = local_extract_path / "workflow_resstock.osw"
    shutil.copy(WORKFLOW_OG, osw_dst)
    with open(osw_dst, "r") as f:
        data = json.load(f)
    data["seed_file"] = "in.osm"
    data["weather_file"] = WEATHER_FILE
    with open(osw_dst, "w") as f:
        json.dump(data, f, indent=2)

def building_done(local_extract_path: Path) -> bool:
    # crude skip: consider done if models dir exists and has files
    models = local_extract_path / "models"
    return models.exists() and any(models.iterdir())

def process_building(bid: int):
    padded = f"{bid:07d}"
    filename = f"bldg{padded}{MODEL_ZIP_SUFFIX}"
    s3_url = S3_BASE + filename
    local_extract_path = EXTRACT_DIR / f"bldg{padded}"
    models_dir = local_extract_path / "models"
    zip_file = local_extract_path / filename

    if building_done(local_extract_path):
        return bid, "skipped"

    try:
        download_s3(s3_url, zip_file)
        unzip_to_dir(zip_file, models_dir)
        copy_weather_and_measures(local_extract_path)
        patch_osw(local_extract_path)
        # zip_file.unlink(missing_ok=True)  # optionally delete zip
        return bid, "ok"
    except zipfile.BadZipFile as e:
        return bid, f"bad_zip: {e}"
    except subprocess.CalledProcessError as e:
        return bid, f"download_fail: {e}"
    except Exception as e:
        return bid, f"error: {e}"

# ========================
# MAIN (parallel)
# ========================
def main():
    ensure_base_files()
    print(f"Upgrade {UPGRADE}: {S3_BASE}  ->  {EXTRACT_DIR}")
    print(f"Using downloader: {'boto3 (multipart)' if _use_boto3 else 'aws cli module'}")
    results = {"ok": [], "skipped": [], "failed": []}

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(process_building, bid): bid for bid in BUILDING_IDS}
        for fut in as_completed(futures):
            bid, status = fut.result()
            if status == "ok":
                print(f"✔ {bid} done")
                results["ok"].append(bid)
            elif status == "skipped":
                print(f"↩ {bid} skipped (already processed)")
                results["skipped"].append(bid)
            else:
                print(f"✖ {bid} {status}")
                results["failed"].append((bid, status))

    print("\nSUMMARY")
    print("-------")
    print(f"Completed: {results['ok']}")
    print(f"Skipped:   {results['skipped']}")
    print(f"Failed:    {results['failed']}")

if __name__ == "__main__":
    main()


Upgrade 4: s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2024/resstock_amy2018_release_2/model_and_schedule_files/building_energy_models/upgrade=4/  ->  c:\Users\jxiong\Documents\GitHub\grid_edge_ithaca\grid_edge_fy25\notebooks\..\data\resstock_models_up04
Using downloader: boto3 (multipart)
↩ 6232 skipped (already processed)
↩ 3424 skipped (already processed)
↩ 1700 skipped (already processed)
↩ 510 skipped (already processed)
↩ 11490 skipped (already processed)
↩ 7612 skipped (already processed)
↩ 6034 skipped (already processed)
↩ 10735 skipped (already processed)
↩ 7830 skipped (already processed)
↩ 12010 skipped (already processed)
↩ 15799 skipped (already processed)
↩ 12342 skipped (already processed)
↩ 15323 skipped (already processed)
↩ 13761 skipped (already processed)
↩ 19388 skipped (already processed)
↩ 21336 skipped (already processed)
↩ 17024 skipped (already processed)
↩ 25421 skipped (already processed)
↩ 19791 skipped (already 

---
## 3. Run OpenStudio simulations
Parallel run (Windows; edit `openstudio_path` if OpenStudio 3.8.0 is installed elsewhere). Needs `building_ids` (from 2a). One simulation per CPU core (`MAX_PARALLEL`). A building counts as failed if the exit code is non-zero or `run/failed.job` exists; the summary prints the first Severe error from each failure's `run/eplusout.err` (full details in `bldgXXXXXXX/run/run.log`), grouped by cause. Re-running the cell skips buildings that already finished (`SKIP_FINISHED`) and first copies the current `base_files/measures/add_outputs/measure.rb` into every model (`SYNC_MEASURES`).

*Known issue (fixed in `add_outputs`):* earlier versions added EMS sensors for meters that EnergyPlus only creates when a matching component exists (`WaterSystems:NaturalGas`, `Cooling:Electricity` for homes without electric cooling, ...). A missing meter aborts the run with `Invalid Output:Variable or Output:Meter ... Unique Key Name not found`. Now only `Electricity:Facility` (always present, and the only one the load tables use) is registered by default; end-use, PV and fuel meters are opt-in via `include_extra_meters` (default false).


In [13]:
import os
import re
import shutil
import subprocess
from collections import defaultdict
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

openstudio_path = "C:/openstudio-3.8.0/bin/openstudio.exe"   # edit if installed elsewhere

# Quick check
subprocess.run([openstudio_path, "openstudio_version"], check=True, capture_output=True, text=True)

MAX_PARALLEL = os.cpu_count() or 8   # one EnergyPlus process per core; 50 at once just thrashes
SKIP_FINISHED = True                 # skip buildings that already have run/eplusout.csv and no failed.job
SYNC_MEASURES = True                 # copy the current add_outputs/measure.rb from data/base_files into every model
extract_dir = Path(extract_dir)      # data/resstock_models[_upNN], from the "Upgrade selection" cell
measures_src = Path("../data/base_files/measures/add_outputs/measure.rb")

def first_error(run_dir: Path) -> str:
    """First '** Severe **' block of eplusout.err (the real cause), else the last ERROR/FATAL line of run.log."""
    err = run_dir / "eplusout.err"
    if err.exists():
        lines = err.read_text(errors="replace").splitlines()
        for i, l in enumerate(lines):
            if "** Severe" in l or "**  Fatal" in l:
                block = [re.sub(r"^\s*\*\*\s*(~~~|Severe|Fatal)\s*\*\*\s*", "", x).strip() for x in lines[i:i + 4]]
                return " ".join(b for b in block if b)[:250]
    log = run_dir / "run.log"
    if log.exists():
        lines = [l for l in log.read_text(errors="replace").splitlines() if "ERROR" in l or "FATAL" in l]
        if lines:
            return lines[-1][:250]
    return ""

def run_one(bid: int):
    padded = f"{bid:07d}"
    bdir = Path(extract_dir) / f"bldg{padded}"
    osw = bdir / "workflow_resstock.osw"
    run_dir = bdir / "run"
    if not osw.exists():
        return bid, "MISSING", f"Missing OSW: {osw}"
    if SKIP_FINISHED and (run_dir / "eplusout.csv").exists() and not (run_dir / "failed.job").exists():
        return bid, "SKIP", ""
    if SYNC_MEASURES:
        shutil.copyfile(measures_src, bdir / "measures" / "add_outputs" / "measure.rb")
    shutil.rmtree(run_dir, ignore_errors=True)   # clear leftovers (e.g. failed.job) from earlier attempts
    r = subprocess.run([openstudio_path, "run", "-w", str(osw)], capture_output=True, text=True)
    # Success = exit code 0 and no failed.job. OpenStudio writes deprecation warnings to stderr, so stderr is not a failure signal.
    ok = r.returncode == 0 and not (run_dir / "failed.job").exists()
    if ok:
        return bid, "OK", ""
    return bid, "FAIL", first_error(run_dir) or r.stderr.strip()[-200:] or f"rc={r.returncode}"

print(f"Upgrade {UPGRADE}: running {len(building_ids)} models from {extract_dir} (up to {MAX_PARALLEL} at once)...")
results = {"OK": [], "SKIP": [], "MISSING": [], "FAIL": []}

with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
    futures = [ex.submit(run_one, bid) for bid in building_ids]
    for i, fut in enumerate(as_completed(futures), 1):
        bid, status, msg = fut.result()
        results[status].append((bid, msg))
        print(f"[{i}/{len(building_ids)}] bldg {bid}: {status}" + (f"  {msg}" if status in ("FAIL", "MISSING") else ""))

print({k: len(v) for k, v in results.items()})
if results["FAIL"] or results["MISSING"]:
    by_cause = defaultdict(list)
    for bid, msg in results["FAIL"] + results["MISSING"]:
        by_cause[msg].append(bid)
    print("\nFailed / missing buildings, grouped by first error:")
    for msg, ids in sorted(by_cause.items(), key=lambda kv: -len(kv[1])):
        print(f"  [{len(ids)}] {msg}\n      {sorted(ids)[:15]}{' ...' if len(ids) > 15 else ''}")
else:
    print("All done")


Upgrade 4: running 483 models from c:\Users\jxiong\Documents\GitHub\grid_edge_ithaca\grid_edge_fy25\notebooks\..\data\resstock_models_up04 (up to 22 at once)...
[1/483] bldg 6034: OK
[2/483] bldg 17024: OK
[3/483] bldg 12342: OK
[4/483] bldg 3424: OK
[5/483] bldg 7830: OK
[6/483] bldg 19388: OK
[7/483] bldg 15323: OK
[8/483] bldg 26305: OK
[9/483] bldg 13761: OK
[10/483] bldg 23725: OK
[11/483] bldg 7612: OK
[12/483] bldg 19791: OK
[13/483] bldg 25421: OK
[14/483] bldg 11490: OK
[15/483] bldg 15799: OK
[16/483] bldg 1700: OK
[17/483] bldg 6232: OK
[18/483] bldg 510: OK
[19/483] bldg 29214: OK
[20/483] bldg 12010: OK
[21/483] bldg 10735: OK
[22/483] bldg 21336: OK
[23/483] bldg 31170: OK
[24/483] bldg 35139: OK
[25/483] bldg 31538: OK
[26/483] bldg 34365: OK
[27/483] bldg 46345: OK
[28/483] bldg 34358: OK
[29/483] bldg 43492: OK
[30/483] bldg 54086: OK
[31/483] bldg 36179: OK
[32/483] bldg 34198: OK
[33/483] bldg 41042: OK
[34/483] bldg 35310: OK
[35/483] bldg 41951: OK
[36/483] bldg 45

---
## 4. Build ResStock load tables for the optimization
Reads `eplusout.csv` (column `EMS:EMS_ElectricityFacility_kWh [](TimeStep)`) from each building in `extract_dir` and writes 15-min, hourly and annual tables to `data/parquet/` (CSV + Parquet), named `resstock_timeseries_all_15[_upNN]`, `..._all_hour[_upNN]` and `..._annual[_upNN]`. Buildings are read in parallel and streamed into the combined files, and existing output files for the selected upgrade are overwritten. Known TODO: make sure Feb 29 is excluded (models run as a non-leap year).


In [7]:
# RESSTOCK DATA PREPARATION FOR OPTIMIZATION


## SCRIPT FOR GENERATING 15 MIN, 1 HOUR PARQUET FILES FROM EPLUSOUT.CSV RESSTOCK SIMULATIONS- PARALLELIZED

# --- ResStock: Build combined 15-min, hourly, and annual tables (no per-building files) ---
# Parallel read; stream-append to combined CSVs; Parquet via pyarrow if available.

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd


# ========================
# CONFIG
# ========================
# extract_dir (resstock_models[_upNN]) and OUT_15_*/OUT_HR_*/OUT_ANL_* (..._upNN.csv/.parquet)
# all come from the "Upgrade selection" cell.
extract_dir = Path(extract_dir)  # where bldgXXXXXX folders live
out_dir     = Path(parquet_folder)
out_dir.mkdir(parents=True, exist_ok=True)

YEAR_PREFIX = "2024-"
N_WORKERS   = 8  # tune for CPU/SSD

DATE_COL = "Date/Time"
TS_COL_CANDIDATES = [
    "EMS:EMS_ElectricityFacility_kWh [](TimeStep)"
]

# Speed up CSV parsing if pyarrow is available
READ_KW = {"usecols": [DATE_COL] + TS_COL_CANDIDATES}
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    READ_KW["engine"] = "pyarrow"
    parquet_ok = True
except Exception:
    parquet_ok = False

# ------------------------
# Small I/O helpers
# ------------------------
def append_csv(df: pd.DataFrame, path: Path, columns_order: list[str]):
    path.parent.mkdir(parents=True, exist_ok=True)
    # enforce column order and append with header once
    df = df.loc[:, columns_order]
    header = not path.exists()
    df.to_csv(path, index=False, mode="a", header=header)

class StreamParquetWriter:
    """Append rows to a single Parquet file using pyarrow ParquetWriter."""
    def __init__(self, path: Path, schema_fields: list[tuple[str, pa.DataType]]):
        self.path = path
        self.schema = pa.schema([pa.field(n, t) for n, t in schema_fields])
        self.writer = None

    def write(self, df: pd.DataFrame):
        if df.empty:
            return
        tbl = pa.Table.from_pandas(df, schema=self.schema, preserve_index=False)
        if self.writer is None:
            self.writer = pq.ParquetWriter(str(self.path), schema=self.schema, compression="snappy")
        self.writer.write_table(tbl)

    def close(self):
        if self.writer is not None:
            self.writer.close()

# ------------------------
# Parsing helpers
# ------------------------
def find_bldg_id_from_path(p: Path) -> int | None:
    for parent in [p] + list(p.parents):
        name = parent.name
        if name.startswith("bldg") and name[4:].isdigit():
            return int(name[4:])
    return None

def pick_ts_column(df_cols) -> str | None:
    for c in TS_COL_CANDIDATES:
        if c in df_cols:
            return c
    return None

def parse_time_col(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip().str.replace(r"\s{2,}", " ", regex=True)
    s = s.str.replace("24:00:00", "00:00:00", regex=False)
    return pd.to_datetime(YEAR_PREFIX + s, format="%Y-%m/%d %H:%M:%S", errors="coerce")

# ------------------------
# Per-file processing
# ------------------------
def process_csv(csv_path: Path):
    """Returns (bldg_id, df15, df_hr, ann_kwh, status) for one eplusout.csv."""
    bldg_id = find_bldg_id_from_path(csv_path)
    if bldg_id is None:
        return (-1, None, None, 0.0, f"skip:no_bldg_id:{csv_path}")

    # Read minimal columns
    try:
        df = pd.read_csv(csv_path, **READ_KW)
    except ValueError as e:
        return (bldg_id, None, None, 0.0, f"missing_cols:{e}")
    except Exception as e:
        return (bldg_id, None, None, 0.0, f"read_fail:{e}")

    ts_col = pick_ts_column(df.columns)
    if ts_col is None:
        return (bldg_id, None, None, 0.0, "no_ts_col")

    # Normalize
    df = df.rename(columns={DATE_COL: "time", ts_col: "ts_elec"})
    df["time"] = parse_time_col(df["time"]).astype("datetime64[ns]")
    df = df.dropna(subset=["time"])
    df["ts_elec"] = pd.to_numeric(df["ts_elec"], errors="coerce").fillna(0.0)

    # 15-min rows for this building
    df15 = df.loc[:, ["time", "ts_elec"]].copy()
    df15.insert(0, "bldg_id", bldg_id)

    # Hourly aggregate for this building
    df_hr = (
        df.assign(time=df["time"].dt.floor("h"))
          .groupby("time", as_index=False)["ts_elec"].sum()
    )
    df_hr.insert(0, "bldg_id", bldg_id)

    ann_kwh = float(df["ts_elec"].sum())
    return (bldg_id, df15, df_hr, ann_kwh, "ok")

# ------------------------
# Discover all CSVs
# ------------------------
print(f"Upgrade {UPGRADE}: reading {extract_dir}")
csv_paths = []
for bdir in extract_dir.iterdir():
    if bdir.is_dir():
        p = next(bdir.rglob("eplusout.csv"), None)
        if p:
            csv_paths.append(p)
        else:
            print(f"No eplusout.csv in {bdir}")
print(f"Found {len(csv_paths)} eplusout.csv files")

# Overwrite any existing combined CSVs (remove so headers re-write cleanly)
for p in [OUT_15_CSV, OUT_HR_CSV, OUT_ANL_CSV]:
    if p.exists():
        p.unlink()

# Prepare Parquet writers (if available)
if parquet_ok:
    writer_15 = StreamParquetWriter(
        OUT_15_PQ,
        [("bldg_id", pa.int64()), ("time", pa.timestamp("ns")), ("ts_elec", pa.float64())],
    )
    writer_hr = StreamParquetWriter(
        OUT_HR_PQ,
        [("bldg_id", pa.int64()), ("time", pa.timestamp("ns")), ("ts_elec", pa.float64())],
    )
else:
    writer_15 = writer_hr = None
    print("[warn] pyarrow not available; writing only CSVs. Install: python -m pip install pyarrow")

annual_rows = []

# ------------------------
# Parallel parse + stream writes
# ------------------------
cols_order = ["bldg_id", "time", "ts_elec"]

with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futs = [pool.submit(process_csv, p) for p in csv_paths]
    for fut in as_completed(futs):
        bldg_id, df15, df_hr, ann_kwh, status = fut.result()
        if status == "ok":
            # Append to CSVs
            append_csv(df15, OUT_15_CSV, cols_order)
            append_csv(df_hr, OUT_HR_CSV, cols_order)
            # Append to Parquet (if available)
            if parquet_ok:
                writer_15.write(df15)
                writer_hr.write(df_hr)
            # Accumulate annual
            annual_rows.append((bldg_id, ann_kwh))
            print(f" {bldg_id} OK")
        elif status == "skipped":
            print(f" {bldg_id} skipped")
        else:
            print(f" {bldg_id} {status}")

# Close parquet writers
if parquet_ok:
    writer_15.close()
    writer_hr.close()

# ------------------------
# Annual combined tables
# ------------------------
anl_df = (
    pd.DataFrame(annual_rows, columns=["bldg_id", "annl_elec"])
      .sort_values("bldg_id")
)
anl_df.to_csv(OUT_ANL_CSV, index=False)

if parquet_ok:
    tbl = pa.Table.from_pandas(anl_df, preserve_index=False)
    pq.write_table(tbl, OUT_ANL_PQ)

print("\nSaved:")
print(f"  15-min  {OUT_15_CSV}")
if parquet_ok: print(f"            {OUT_15_PQ}")
print(f"  Hourly   {OUT_HR_CSV}")
if parquet_ok: print(f"            {OUT_HR_PQ}")
print(f"  Annual   {OUT_ANL_CSV}")
if parquet_ok: print(f"            {OUT_ANL_PQ}")
print("DONE.")


Upgrade 4: reading c:\Users\jxiong\Documents\GitHub\grid_edge_ithaca\grid_edge_fy25\notebooks\..\data\resstock_models_up04
Found 483 eplusout.csv files
 510 OK
 6232 OK
 3424 OK
 10735 OK
 6034 OK
 1700 OK
 7830 OK
 7612 OK
 15323 OK
 13761 OK
 19388 OK
 11490 OK
 17024 OK
 12010 OK
 15799 OK
 12342 OK
 31538 OK
 23725 OK
 25421 OK
 21336 OK
 29214 OK
 19791 OK
 26305 OK
 31170 OK
 34358 OK
 36179 OK
 37051 OK
 41042 OK
 34198 OK
 35139 OK
 34365 OK
 35310 OK
 41951 OK
 43246 OK
 43492 OK
 43442 OK
 44766 OK
 46345 OK
 45739 OK
 50186 OK
 53086 OK
 50361 OK
 54086 OK
 56299 OK
 55788 OK
 55240 OK
 55068 OK
 57646 OK
 57693 OK
 58658 OK
 58681 OK
 60411 OK
 59156 OK
 62476 OK
 61315 OK
 62164 OK
 63725 OK
 62706 OK
 65865 OK
 67981 OK
 69315 OK
 64522 OK
 68088 OK
 70027 OK
 71034 OK
 71681 OK
 73290 OK
 70312 OK
 72775 OK
 76867 OK
 78956 OK
 78512 OK
 79120 OK
 80655 OK
 83809 OK
 84195 OK
 85261 OK
 84753 OK
 84430 OK
 86623 OK
 87415 OK
 88300 OK
 87859 OK
 87594 OK
 88748 OK
 92289